# Fusion-JEPA: Supercomputer, Kaggle & Colab Pipeline

This interactive notebook automates the entire **Fusion-JEPA** workflow:
1. Cloning the latest repository and BigVGAN neural vocoder submodule.
2. Installing environment dependencies.
3. Automated dataset downloading and PyTorch Lightning training.
4. Audio generation (Inference) with immediate **in-notebook audio playback**.

### 1. Check GPU Accelerator

In [ ]:
!nvidia-smi

### 2. Clone Repository & Initialize Submodules

In [ ]:
# Clone the latest Fusion-JEPA repository with submodules
!git clone --recursive https://github.com/OmarA32/Fusion-JEPA-TTS.git
%cd Fusion-JEPA-TTS
!git checkout main
!git submodule update --init --recursive

### 3. Install Dependencies
Installs PyTorch Lightning, BigVGAN dependencies, and Arabic/English phonetizers.

In [ ]:
!pip install -r requirements.txt

### 4. Hugging Face Authentication (Optional)
If you want to automatically upload checkpoint weights during training to Hugging Face, paste your Write token below.

In [ ]:
import json

HF_TOKEN = "" # Paste your Hugging Face write token here (e.g. hf_xxxxx)

if HF_TOKEN.startswith("hf_") and len(HF_TOKEN) > 10:
    with open("hf_config.json", "w") as f:
        json.dump({"HF_TOKEN": HF_TOKEN}, f)
    print("Token saved! The training script will automatically upload checkpoints.")
else:
    print("No token provided. Running in local saving mode.")

### 5. Download Pre-Trained Weights (Optional)
Fetches the latest model checkpoint from Hugging Face to resume training or run inference.

In [ ]:
# Download Arabic or English weights
!python download_from_hf.py --lang arabic
# !python download_from_hf.py --lang english

### 5B. Pre-Download LJSpeech-1.1 Dataset (Optional / For English)
Downloads and fast-extracts the 13,100 LJSpeech audio clips directly into `data/LJSpeech-1.1` on Kaggle.

In [ ]:
import os

os.makedirs("data", exist_ok=True)
ljs_dir = "data/LJSpeech-1.1"

if not os.path.exists(ljs_dir):
    print("Downloading LJSpeech-1.1 dataset (2.6 GB)...")
    !wget -q --show-progress https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2 -P data/
    
    print("Extracting LJSpeech-1.1...")
    !tar -xjf data/LJSpeech-1.1.tar.bz2 -C data/
    
    if os.path.exists("data/LJSpeech-1.1.tar.bz2"):
        os.remove("data/LJSpeech-1.1.tar.bz2")
    print("LJSpeech-1.1 is ready in data/LJSpeech-1.1!")
else:
    print("LJSpeech-1.1 is already present in data/LJSpeech-1.1!")


### 6A. Train Model (Arabic or English)
Launches PyTorch Lightning training with dual Continuous Flow Matching (Lv) and JEPA Latent Prediction (Lp).

In [ ]:
# Arabic Training on Nawar Halabi dataset:
!python train.py --lang arabic --db nawar_halabi --checkpointnum 5

# English Training on LJSpeech dataset:
# !python train.py --lang english --db ljspeech --checkpointnum 5

### 6B. Overfitting Verification Protocol (Quick Sanity Check)
Trains on a single speech sample to verify that the MM-DiT backbone rapidly reconstructs formants and eliminates representation collapse.

In [ ]:
!python overfit_test.py --lang arabic --epochs 500
# !python overfit_test.py --lang english --epochs 500

### 7. Speech Synthesis (Inference) & In-Notebook Audio Player
Synthesizes high-fidelity 44.1 kHz speech using the MM-DiT flow predictor and BigVGAN v2 vocoder.

In [ ]:
# Synthesis Configuration
file_name = "arabic_demo.wav"
text = "وَتَتَضَمَّنُ حَفَلَاتٍ لِمُوسِيقَى الْجَازِ"

# 1. Generate Arabic speech
!python inference.py --lang arabic --text "$text" --output "$file_name" --save-mel

# 2. Play the generated audio directly inside the notebook:
import IPython.display as ipd
ipd.Audio(f"test_results/{file_name}")


### 8. English Speech Synthesis

In [ ]:
# Synthesis Configuration
file_name = "english_demo.wav"
text = "This is Fusion JEPA text to speech synthesis."

# 1. Generate English speech
!python inference.py --lang english --text "$text" --output "$file_name" --save-mel

# 2. Play the generated audio directly inside the notebook:
import IPython.display as ipd
ipd.Audio(f"test_results/{file_name}")


### 9. Long-Form Arabic Speech Synthesis
Synthesizes long Arabic paragraphs of arbitrary duration using prosodic clause chunking and seamless acoustic breath-pause stitching.

In [ ]:
# Long-Form Arabic Paragraph Synthesis
file_name = "arabic_longform_demo.wav"
text = "يَعْتَمِدُ نِظَامُ فُيُوجِن جِيبَا عَلَى التَّعَلُّمِ الذَّاتِيِّ لِتَوْلِيدِ صَوْتٍ عَالِي الْجَوْدَةِ، وَيَتَمَيَّزُ بِقُدْرَتِهِ عَلَى مُعَالَجَةِ النُّصُوصِ الْعَرَبِيَّةِ الْمُعَقَّدَةِ بِكُلِّ دِقَّةٍ وَوُضُوحٍ."

# 1. Generate Long-Form Arabic speech
!python longform_inference.py --lang arabic --text "$text" --output "$file_name" --save-mel

# 2. Play the generated long audio directly inside the notebook:
import IPython.display as ipd
ipd.Audio(f"test_results/{file_name}")

### 10. Long-Form English Speech Synthesis
Synthesizes multi-sentence English paragraphs of arbitrary duration with natural intonation, zero canvas overflow, and zero clicks.

In [ ]:
# Long-Form English Paragraph Synthesis
file_name = "english_longform_demo.wav"
text = "Fusion-JEPA is a deep multimodal architecture designed for expressive text-to-speech synthesis. Its also bilingual and can work with arabic and english."

# 1. Generate Long-Form English speech
!python longform_inference.py --lang english --text "$text" --output "$file_name" --save-mel

# 2. Play the generated long audio directly inside the notebook:
import IPython.display as ipd
ipd.Audio(f"test_results/{file_name}")

### 11. Test BigVGAN Ground-Truth Vocoding Quality
Reconstructs an uncompressed 44.1 kHz audio waveform directly from a ground-truth dataset Mel-spectrogram.

In [ ]:
!python test_vocoder_ground_truth.py --lang arabic --db nawar_halabi --index 10

import IPython.display as ipd
ipd.Audio("test_results/ground_truth_index_10_bigvgan.wav")